In [13]:
import warnings
#warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

from langchain_anthropic import ChatAnthropic
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

import sqlite3
import datetime
import pandas as pd
import httpx
from bs4 import BeautifulSoup

claude = ChatAnthropic(model="claude-sonnet-4-5", temperature=0)
search = DuckDuckGoSearchResults(num_results=5)

print("Ready!")

Ready!


In [14]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(PROJECT_ROOT, "deep-research-agent", "research_history.db")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "deep-research-agent", "reports")

def init_database():
    """Create the database and tables if they don't exist."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS research_runs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT NOT NULL,
            topic TEXT NOT NULL,
            num_searches INTEGER,
            num_pages_scraped INTEGER,
            report_filename TEXT,
            duration_seconds REAL,
            llm_used TEXT,
            status TEXT,
            input_tokens INTEGER DEFAULT 0,
            output_tokens INTEGER DEFAULT 0,
            total_tokens INTEGER DEFAULT 0
        )
    """)
    
    # Migrate existing database if columns don't exist yet
    existing_columns = [
        row[1] for row in cursor.execute("PRAGMA table_info(research_runs)")
    ]
    for col in ["input_tokens", "output_tokens", "total_tokens"]:
        if col not in existing_columns:
            cursor.execute(f"ALTER TABLE research_runs ADD COLUMN {col} INTEGER DEFAULT 0")
            print(f"  Added column: {col}")
    
    conn.commit()
    conn.close()
    print(f"Database ready at: {DB_PATH}")

init_database()

  Added column: input_tokens
  Added column: output_tokens
  Added column: total_tokens
Database ready at: C:\Users\Henry\Documents\deep-research-agent\research_history.db


In [15]:
#Build the research logging tool
def log_research_run(topic, num_searches, num_pages_scraped,
                     report_filename, duration_seconds, llm_used, status,
                     input_tokens=0, output_tokens=0):
    """Log a completed research run to SQLite."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        INSERT INTO research_runs 
        (timestamp, topic, num_searches, num_pages_scraped, 
         report_filename, duration_seconds, llm_used, status,
         input_tokens, output_tokens, total_tokens)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.datetime.now().isoformat(),
        topic,
        num_searches,
        num_pages_scraped,
        report_filename,
        duration_seconds,
        llm_used,
        status,
        input_tokens,
        output_tokens,
        input_tokens + output_tokens
    ))
    
    conn.commit()
    conn.close()
    return cursor.lastrowid

print("Logging function ready!")

Logging function ready!


In [30]:
# Claude agent initialization
claude = ChatAnthropic(model="claude-sonnet-4-5", temperature=0)

agent = create_react_agent(claude, tools=tools)

print("✅ Agent ready!")

C:\Users\Henry\AppData\Local\Temp\ipykernel_29716\2721661649.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(claude, tools=tools)


✅ Agent ready!


In [31]:
import time
import re

def run_research(topic: str, prompt_template: str = None):
    """Run research with Claude and track token usage."""
    
    run_metrics["num_searches"] = 0
    run_metrics["num_pages_scraped"] = 0
    run_metrics["scraped_urls"] = []
    
    if not prompt_template:
        prompt_template = f"""
        You are a thorough research agent. Follow these steps carefully:
        
        1. Search for '{topic}' — do this 3 to 4 times with different queries
        
        2. From all the URLs you find, select the 3 to 4 MOST RELEVANT and CREDIBLE ones:
           - Prefer: official sources, academic sites, reputable news outlets, industry blogs
           - Avoid: forums, social media, low quality blogs, paywalled content
        
        3. Scrape each selected page and evaluate whether it contains genuinely useful 
           information. Skip any page that is too thin or irrelevant.
        
        4. Synthesize your findings into a structured report with these sections:
           - Executive Summary
           - Key Findings (cite the source URL next to each finding)
           - Detailed Analysis
           - Sources (only URLs you actually visited and found useful)
           - Conclusion
           You can add additional section if it is deemed informative and important.
        
        5. Save the report with a descriptive filename related to '{topic}'
        
        Important: Never include sources you did not actually visit and scrape.
        You must complete all steps in under 20 tool calls total.
        """
    
    start_time = time.time()
    status = "success"
    report_filename = ""
    input_tokens = 0
    output_tokens = 0
    last_chunk = None

    print(f"🚀 Starting research on: '{topic}'", flush=True)
    print("-" * 50, flush=True)

    try:
        for chunk in agent.stream(
            {"messages": [HumanMessage(content=prompt_template)]},
            config={"recursion_limit": 25}
        ):
            last_chunk = chunk
            elapsed = round(time.time() - start_time, 1)

            if "agent" in chunk:
                for message in chunk["agent"].get("messages", []):
                    content = message.content
                    
                    # Extract token usage from each Claude response
                    usage = message.response_metadata.get("usage", {})
                    input_tokens += usage.get("input_tokens", 0)
                    output_tokens += usage.get("output_tokens", 0)
                    
                    if isinstance(content, list):
                        for block in content:
                            if hasattr(block, "name"):
                                print(f"[{elapsed}s] 🤖 Calling tool → {block.name}", flush=True)
                                if hasattr(block, "input"):
                                    print(f"         Input: {str(block.input)[:120]}", flush=True)
                    elif isinstance(content, str) and content.strip():
                        print(f"[{elapsed}s] 🤖 Claude writing response...", flush=True)
                        print(f"         Preview: {content[:150]}...", flush=True)

            elif "tools" in chunk:
                for message in chunk["tools"].get("messages", []):
                    tool_name = getattr(message, "name", "unknown")
                    print(f"[{elapsed}s] 🔧 [{tool_name}] returned results", flush=True)

        # Extract report filename from last message
        if last_chunk and "agent" in last_chunk:
            messages = last_chunk["agent"].get("messages", [])
            if messages:
                final_content = messages[-1].content
                if isinstance(final_content, str):
                    match = re.search(r'[\w_-]+\.md', final_content)
                    if match:
                        report_filename = match.group(0)

    except Exception as e:
        status = f"error: {str(e)}"
        print(f"\n❌ Error: {e}", flush=True)

    duration = round(time.time() - start_time, 2)

    # Log to SQLite with token counts
    run_id = log_research_run(
        topic=topic,
        num_searches=run_metrics["num_searches"],
        num_pages_scraped=run_metrics["num_pages_scraped"],
        report_filename=report_filename,
        duration_seconds=duration,
        llm_used="claude-sonnet-4-6",
        status=status,
        input_tokens=input_tokens,
        output_tokens=output_tokens
    )

    print("\n" + "-" * 50, flush=True)
    print(f"✅ Research complete!", flush=True)
    print(f"   Topic: {topic}", flush=True)
    print(f"   Searches made: {run_metrics['num_searches']}", flush=True)
    print(f"   Pages scraped: {run_metrics['num_pages_scraped']}", flush=True)
    print(f"   Duration: {duration}s", flush=True)
    print(f"   Claude tokens — Input: {input_tokens:,} | Output: {output_tokens:,} | Total: {input_tokens + output_tokens:,}", flush=True)
    print(f"   Run ID: {run_id}", flush=True)

    return last_chunk

# Run it!
result = run_research("future of renewable energy 2025")

🚀 Starting research on: 'future of renewable energy 2025'
--------------------------------------------------
[7.7s] 🔧 [search_web] returned results
[7.8s] 🔧 [search_web] returned results
[10.0s] 🔧 [search_web] returned results
[12.6s] 🔧 [search_web] returned results
[20.9s] 🔧 [scrape_webpage] returned results
[21.1s] 🔧 [scrape_webpage] returned results
[21.2s] 🔧 [scrape_webpage] returned results
[23.0s] 🔧 [scrape_webpage] returned results
[31.2s] 🔧 [scrape_webpage] returned results
[40.9s] 🔧 [scrape_webpage] returned results
[45.9s] 🔧 [search_web] returned results
[51.2s] 🔧 [scrape_webpage] returned results
[51.5s] 🔧 [scrape_webpage] returned results
[117.3s] 🔧 [save_report] returned results
[129.7s] 🤖 Claude writing response...
         Preview: Perfect! I have successfully completed the comprehensive research on the future of renewable energy in 2025. Here's a summary of what I accomplished:
...

--------------------------------------------------
✅ Research complete!
   Topic: future

In [32]:
def load_history() -> pd.DataFrame:
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM research_runs", conn)
    conn.close()
    return df

df = load_history()

print("=== Research History ===")
print(df[["id", "timestamp", "topic", "num_searches", "duration_seconds", "status"]].to_string())

print("\n=== Token Usage (Claude only) ===")
print(df[["id", "topic", "input_tokens", "output_tokens", "total_tokens"]].to_string())

print("\n=== Summary Stats ===")
print(f"Total runs:               {len(df)}")
print(f"Successful runs:          {len(df[df['status'] == 'success'])}")
print(f"Avg duration:             {df['duration_seconds'].mean():.1f}s")
print(f"Avg searches per run:     {df['num_searches'].mean():.1f}")
print(f"Total Claude tokens used: {df['total_tokens'].sum():,}")
print(f"Avg tokens per run:       {df['total_tokens'].mean():.0f}")

=== Research History ===
    id                   timestamp                                       topic  num_searches  duration_seconds                                                                                                                                                                                                                                                                                                                                                                                                                                   status
0    1  2026-02-27T09:12:35.142459                                  Test topic             3             45.20                                                                                                                                                                                                                                                                                                                                         